# Week 2 ML test

## Data

### Data Loading

In [ ]:
import gc
gc.collect()

In [ ]:
# import tensorflow as tf
from tensorflow.keras.layers import Dense, Input, Dropout#, Normalization, StringLookup, CategoryEncoding, Concatenate
from tensorflow.keras.models import load_model, Sequential
from tensorflow.keras import callbacks, Input, optimizers, regularizers
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import os
import matplotlib.pyplot as plt
# import optuna
# from optuna.integration import TFKerasPruningCallback

# Load and inspect the dataset
data = pd.DataFrame(np.genfromtxt('../Week1/AlephBtag_MC_train_Nev50000.csv', names=True))
display(data.head())
display(data.describe())

# Split the data into features and target (and validation for hyperparams)
variables = data.columns
input_variables = variables[(variables != 'nnbjet') & (variables != 'isb') & (variables != 'energy') & (variables != 'cTheta') & (variables != 'phi')]
X      = data[input_variables]
y = data['isb']
benchmark_data  = data['nnbjet']

# Split into training and test sets
X_train, X_test, y_train, y_test, bench_train, bench_test,  = train_test_split(X, y, benchmark_data, test_size=0.25, random_state=42, shuffle=True)


y_test = np.array(y_test).reshape(-1, 1)
y_train = np.array(y_train).reshape(-1, 1)
# # Identify feature types
# numeric_features = X_train.select_dtypes(include=['int64', 'float64']).columns.tolist()
# categorical_features = X_train.select_dtypes(include=['object', 'category']).columns.tolist()


## TF MWE

In [ ]:
# Define log directory
log_dir = "./logs"
checkpoint_path = log_dir + "/aleph_best_model.keras"  
os.makedirs(log_dir, exist_ok=True)

# Set up callbacks
early_stopping = callbacks.EarlyStopping(
    monitor='val_loss',
    patience=20,
    restore_best_weights=True,
    verbose=1
)

checkpoint = callbacks.ModelCheckpoint(
    filepath=checkpoint_path,
    monitor='val_loss',
    save_best_only=True,
    save_weights_only=False,  # Save the full model
    verbose=1
)

tensorboard = callbacks.TensorBoard(
    log_dir=log_dir,
    histogram_freq=1,
    write_graph=True,
    update_freq='epoch'
)

reduce_lr = callbacks.ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    # patience=10,
    verbose=1,
    # min_lr=1e-6
)

csv_logger = callbacks.CSVLogger(
    filename='./logs/training_log.csv',
    append=True
)

### Hyperparameter Tuning

In [ ]:

from keras_tuner import Hyperband, BayesianOptimization, HyperParameters
hp = HyperParameters()




def build_model(hp):
    # model type
    model = Sequential()

    # add layers
    model.add(Input(shape=(X_train.shape[1],)))
    
    
     # Tune number of hidden layers
    num_layers = hp.Int('num_layers', 1, 3)
    for i in range(num_layers):
        
        # Tune regularization strength ("prior")
        reg_strength = hp.Float('l2_reg', 1e-6, 1e-2, sampling='log')
        
        model.add(Dense(
            units=hp.Int(f'units', min_value=8, max_value=128, step=16),
            kernel_regularizer=regularizers.l2(reg_strength),
            activation=hp.Choice(f'activation', ['relu', 'tanh', 'elu'])
        ))
        
        # Add tunable dropout after each Dense layer
        model.add(Dropout(
            rate=hp.Float(f'dropout_rate', min_value=0.1, max_value=0.5, step=0.05)
        ))



    # Output layer for binary classification
    model.add(Dense(1, activation='sigmoid', name='output'))
    
    # Tune optimizer and learning rate
    optimizer_choice = hp.Choice(f'optimizer', ['adam', 'sgd', 'rmsprop'])
    lr = hp.Float(f'learning_rate', min_value=1e-4, max_value=1e-2, sampling='log')
    
    if optimizer_choice == 'adam':
        optimizer = optimizers.Adam(learning_rate=lr)
    elif optimizer_choice == 'sgd':
        optimizer = optimizers.SGD(learning_rate=lr)
    else:
        optimizer = optimizers.RMSprop(learning_rate=lr)
    
    model.compile(
        optimizer=optimizer,
        loss='binary_crossentropy',
        metrics=['accuracy']
    )
    return model


tuner = Hyperband(
    build_model,
    objective='val_accuracy',
    # max_epochs=10,
    directory='./logs/keras_tuner',
    project_name='aleph_tuning'
)
# tuner = BayesianOptimization(
#     build_model,
#     objective='val_accuracy',
#     max_trials=10,
#     seed=42,
#     directory='./logs/keras_tuner',
#     project_name='aleph_tuning_bayesian'
# )


# Search for best hyperparameters
tuner.search(X_train, y_train, epochs=50, 
             batch_size=hp.Choice('batch_size', [32, 64, 128]),
             validation_data=(X_test, y_test),
             callbacks = [
                 early_stopping,
                #  tensorboard,
                reduce_lr,
                csv_logger
                ],
             verbose=1,  # Suppress progress bars
             )

best_hps = tuner.get_best_hyperparameters(1)[0]


### Build and retrain the best model

In [ ]:
# 1. Build model from best hyperparameters
model = tuner.hypermodel.build(best_hps)

# 2. Fit the model
history = model.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=50,
    callbacks=[early_stopping,
               checkpoint,
            #    tensorboard,
            #    reduce_lr,
               csv_logger,
               ]
)

# 3. Load the best-performing model (based on val_loss)
model = load_model(checkpoint_path)


In [ ]:
test_loss, test_accuracy = model.evaluate(X_test, y_test, verbose=2)



plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.title('Training History')
plt.show()



### Input Feature Ranking

In [ ]:
import sklearn
import shap
from sklearn.model_selection import train_test_split

# print the JS visualization code to the notebook
shap.initjs()

# train a SVM classifier
# X_train,X_test,Y_train,Y_test = train_test_split(*shap.datasets.iris(), test_size=0.2, random_state=0)
# svm = sklearn.svm.SVC(kernel='rbf', probability=True)
# svm.fit(X_train, Y_train)

# use Kernel SHAP to explain test set predictions
explainer = shap.KernelExplainer(probs, X_train, link="logit")
shap_values = explainer.shap_values(X_test, nsamples=100)

# plot the SHAP values for the Setosa output of the first instance
shap.force_plot(explainer.expected_value[0], shap_values[0][0,:], X_test.iloc[0,:], link="logit")

### old

In [ ]:
# # Build input preprocessing
# from tensorflow.python.data.ops.dataset_ops import DatasetV2
# import keras_tuner


# # - - - - - - - - - - -

# # # Build DNN model
# # x = Dense(128, activation='relu')(concatenated)
# # x = Dropout(0.2)(x)
# # x = Dense(64, activation='relu')(x)
# # outputs = Dense(len(np.unique(y)), activation='softmax')(x)

# model = Sequential([
#     Dense(9,activation='relu',name='input_layer1'),
#     Dense(24,activation='relu',name='hidden_layer1'),
#     Dense(12,activation='relu',name='hidden_layer2'),
#     Dense(1, activation='sigmoid', name='output')])


# # Compile model
# model.compile(optimizer='adam',
#               loss='binary_crossentropy',
#               metrics=['binary_crossentropy'])

# # # Helper: convert dataframe to dict-style tf.data.Dataset
# # def df_to_dataset(X, y, shuffle=True, batch_size=32):
# #     ds = tf.data.Dataset.from_tensor_slices((dict(X), y))
# #     if shuffle:
# #         ds = ds.shuffle(buffer_size=len(X))
# #     return ds.batch(batch_size)

# # # Prepare tf.data datasets
# # batch_size = 32
# # train_ds: DatasetV2 = df_to_dataset(X_train, y_train, batch_size=batch_size)
# # test_ds: DatasetV2 = df_to_dataset(X_test, y_test, shuffle=False, batch_size=batch_size)


# # - - - - - - - - - - -




# # Define log directory
# log_dir = "./logs"
# checkpoint_path = log_dir + "/aleph_best_model.keras"  
# os.makedirs(log_dir, exist_ok=True)

# if os.path.exists(checkpoint_path):
#     print(f"Loading existing model from {checkpoint_path}")
#     model = load_model(checkpoint_path)
# else:
#     # Set up callbacks
#     early_stopping = tf.keras.callbacks.EarlyStopping(
#         monitor='val_loss',
#         patience=20,
#         restore_best_weights=True,
#         verbose=1
#     )

#     checkpoint = tf.keras.callbacks.ModelCheckpoint(
#         filepath=checkpoint_path,
#         monitor='val_loss',
#         save_best_only=True,
#         save_weights_only=False,  # Save the full model
#         verbose=1
#     )
    
#     tensorboard = tf.keras.callbacks.TensorBoard(
#         log_dir=log_dir,
#         histogram_freq=1,
#         write_graph=True,
#         update_freq='epoch'
#     )

#     reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(
#         monitor='val_loss',
#         factor=0.5,
#         patience=10,
#         verbose=1,
#         min_lr=1e-6
#     )

#     csv_logger = tf.keras.callbacks.CSVLogger(
#         filename='./logs/training_log.csv',
#         append=True
#     )

#     # Train model
#     history = model.fit(
#         train_ds,
#         validation_data=test_ds,
#         epochs=100,
#         callbacks = [early_stopping, checkpoint, tensorboard, reduce_lr, csv_logger],
#     )



# # Evaluate
# loss, accuracy = model.evaluate(test_ds)
# print(f"Test accuracy: {accuracy:.4f}")

In [ ]:
model.summary()

## Evaluation

In [ ]:
from sklearn.metrics import roc_curve, auc

# Predict probabilities
y_predict = model.predict(X_test, verbose=0).ravel()

# Compute ROC curve and AUC for your model
fpr, tpr, _ = roc_curve(y_test, y_predict)
auc_score = auc(fpr, tpr)

# Compute ROC and AUC for benchmark
fpr_bench, tpr_bench, _ = roc_curve(y_test, bench_test)
auc_bench = auc(fpr_bench, tpr_bench)

# Plot
fig, ax = plt.subplots(figsize=(10, 10))
ax.plot(fpr, tpr, label=f'Our model (AUC = {auc_score:.3f})')
ax.plot(fpr_bench, tpr_bench, label=f'Aleph NNbjet (AUC = {auc_bench:.3f})')

# Customize
plt.title('Model Comparison (ROC curves)', size=16)
plt.legend(fontsize=16, loc='lower right')
plt.xlabel('False Positive Rate', size=16)
plt.ylabel('True Positive Rate', size=16)
# plt.grid(True)
plt.show()


In [ ]:
from sklearn.metrics import roc_curve
from sklearn.metrics import auc
from sklearn.metrics import confusion_matrix

y_predict = model.predict(X_test)


# Evaluate:
fpr, tpr, _ = roc_curve(y_test, y_predict)                  # False/True Positive Rate for our model
fpr_nnbjet, tpr_nnbjet, _ = roc_curve(y_test, bench_test)  # False/True Positive Rate for Aleph NNbjet

# We can now calculate the Area-Under-the-Curve (AUC) scores of these ROC-curves:
auc_score = auc(fpr,tpr)                        # This is the AUC score for our model
auc_score_nnbjet = auc(fpr_nnbjet, tpr_nnbjet)  # This is the AUC score for Aleph NNbjet

# Let's plot the ROC curves for these results:
fig = plt.figure(figsize = [10,10])
plt.title('Model Comparison (ROC curves)', size = 16)
plt.plot(fpr, tpr, label=f'Our LightGBM model (AUC = {auc_score:5.3f})')
plt.plot(fpr_nnbjet, tpr_nnbjet, label = f'Aleph NNbjet (AUC = {auc_score_nnbjet:5.3f})')
plt.legend(fontsize=16)
plt.xlabel('False Postive Rate', size=16)
plt.ylabel('True Positive Rate', size=16)
plt.show()